# Regex para identificar e extrair padrões

Use regex quando textos seguem uma estrutura. Para uma palavra literal simples, prefira contains(..., regex=False). Regex valida o formato; não comprova que um produto ou endereço existe. r antes da string evita que Python interprete barras como escapes.

Execute as demonstrações; depois tente os exercícios antes do gabarito. Escreva dados disponíveis, resultado esperado e passos em português. Requisitos: Python 3.9 ou superior, pandas e IPython. Para fusos, o ambiente precisa de dados IANA; se houver ZoneInfoNotFoundError, instale tzdata no kernel. Não é necessário baixar datasets.

In [ ]:
import pandas as pd
from IPython.display import display

## Demo Identificação e extração
\d representa dígito; {3} exige três ocorrências; ^ e $ delimitam começo e fim. Parênteses criam grupos de extração; fullmatch exige o texto inteiro.

In [ ]:
produtos = pd.DataFrame({"descricao": ["CAM-001 tamanho M","CAL-023 tamanho G","CAM-104 tamanho P","CAM-12 tamanho M","texto livre",None]})
print(produtos["descricao"].str.contains("CAM-", regex=False, na=False))
padrao = r"(?P<codigo>[A-Z]{3}-\d{3}) tamanho (?P<tamanho>P|M|G)"
extraidos = produtos["descricao"].str.extract(padrao)
display(extraidos)
display(produtos["descricao"].str.fullmatch(padrao, na=False))

## Exercícios

### 1 Extrair e conferir
Extraia código e tamanho somente de descrições que correspondem integralmente ao padrão. Liste registros inválidos, incluindo nulos.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [ ]:
# Sua solução aqui.

### 2 Contar tamanhos
Conte produtos válidos por tamanho e códigos começando por CAM-.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [ ]:
# Sua solução aqui.

### 3 Limpar telefone
Para ["(65) 99999-1234", "+55 11 98888-4321", None], remova caracteres não numéricos preservando ausência. Explique por que isso não valida o telefone.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [ ]:
# Sua solução aqui.

## Gabarito comentado

### 1 Extrair e conferir

Como pensar: Verificar fullmatch → extrair válidos → listar restantes.

In [ ]:
valido = produtos["descricao"].str.fullmatch(padrao, na=False)
resultado = produtos.loc[valido,"descricao"].str.extract(padrao)
display(resultado)
display(produtos.loc[~valido])
assert len(resultado) == 3

### 2 Contar tamanhos

Como pensar: Reutilizar extração válida → contar → filtro literal de prefixo.

In [ ]:
display(resultado["tamanho"].value_counts())
display(resultado.loc[resultado["codigo"].str.startswith("CAM-", na=False)])

### 3 Limpar telefone

Como pensar: \D significa não dígito → substituir por vazio → manter nulo.

In [ ]:
telefones = pd.Series(["(65) 99999-1234", "+55 11 98888-4321", None])
limpos = telefones.str.replace(r"\D", "", regex=True)
display(limpos)
assert limpos.iloc[0] == "65999991234"
assert pd.isna(limpos.iloc[2])

# Oficina de funções e reutilização

Definir prepara uma função; chamar executa seu corpo. Em notebooks, execute primeiro a célula com def. Faça os exercícios antes de consultar as soluções. As oficinas usam dados próprios e podem ser executadas após os imports.

## Demo Limpeza repetida
Temos dois campos com a mesma regra: manter somente dígitos, preservando nulos. Vamos aplicar uma função à coluna inteira.

In [ ]:
celular = pd.Series(["(65) 99999-1234",None])
fixo = pd.Series(["(11) 3333-4444","(41) 2222-3333"])

## Pratique a transformação
Para cada atividade, escreva entrada → processamento → saída. Compare o código antes e depois: onde a regra fica definida? Quantos lugares precisam mudar quando ela muda? Funções não tornam todo código mais rápido; o ganho aqui é organização, reutilização e manutenção.

### 1 Crie a função da coluna

Crie limpar_telefones(coluna) e chame para celular e fixo.

In [ ]:

# Sua solução aqui.

### 2 Função por texto e apply

Crie extrair_codigo(texto) usando re.fullmatch. Retorne o código ou None para nulos e formatos inválidos. Passe a função para apply.

In [ ]:
import re
descricoes = pd.Series(["CAM-001 tamanho M","CAM-12 tamanho M",None])
# Sua solução aqui.

## Soluções da oficina

### 1 Crie a função da coluna

A regra fica única; uma função sobre a Series permite usar o método str diretamente.

In [ ]:
def limpar_telefones(coluna):
    return coluna.str.replace(r"\D","",regex=True)

print(limpar_telefones(celular).tolist())
print(limpar_telefones(fixo).tolist())
assert limpar_telefones(celular).iloc[0] == "65999991234"
assert pd.isna(limpar_telefones(celular).iloc[1])

### 2 Função por texto e apply

Há várias etapas e tratamento de ausência, por isso def é mais claro que uma lambda longa.

In [ ]:
def extrair_codigo(texto):
    if pd.isna(texto):
        return None
    match = re.fullmatch(r"([A-Z]{3}-\d{3}) tamanho (P|M|G)",texto)
    return match.group(1) if match else None

codigos = descricoes.apply(extrair_codigo)
print(codigos.tolist())
assert codigos.iloc[0] == "CAM-001"
assert codigos.iloc[1:].isna().all()

Referência: https://docs.python.org/3/tutorial/controlflow.html#defining-functions . lambda contém uma expressão; use def quando precisar de várias etapas, validação ou uma regra reutilizada com nome.